# API с IGDB

## Контекст:
IGDB - это база данных игр, которая доступна по API.
**Почти тоже самое, что и IMDB, только для игр...**

Мы собираем данные через **5 эндпойнтов**:

|эндпойнт|что собираем|зачем|
|-|-|-|
|games|id, name, slug, first_release_date, platforms, genres, themes, game_modes, game_type, involved_companies, rating, rating_count, aggregated_rating, aggregated_rating_count, total_rating, total_rating_count, hypes, external_games, franchise, parent_game, remakes, remasters, storyline, summary, keywords|основная таблица — ядро датасета|
|involved_companies|company.name, company.id, game.name, game.id (фильтр: developer=true)|связка игра → студия-разработчик|
|keywords|id, name, slug|расшифровка keyword_id в человекочитаемые теги|
|game_time_to_beats|game_id, hastily, normally, completely, count|время прохождения (быстро / нормально / 100%)|
|alternative_names|comment, game, name|альтернативные названия (локализации, аббревиатуры)|

**Фильтр:** `rating != null & rating_count > 20` - берём только игры с достаточным количеством оценок.



Доступ к IGDB возможен только по токену, который мы успешно сгенерировали с помощью Twitch Developer. Подробнее о том, как это сделать можно узнать в [документации IGDB](https://api-docs.igdb.com/#getting-started)

В следующих этапах проекта, мы объединим данные с API со скреппингом (либо используем иначе).

### Начнем с настройки API
Чтобы наши прелестные ключики не улетели в свободное использование на github (так как репозиторий публичен), подтянем их из окружения, до этого закинув их в терминал

In [7]:
import os

In [ ]:
os.environ['twitch_client_secret'] = "6767767676"
os.environ['twitch_client_id'] = "67676767767"

#пж замените на данные в тгшке, но не пуште с секретками

In [47]:
cid = os.environ['twitch_client_id']
secret = os.environ['twitch_client_secret']

In [ ]:
# а теперь получим bearer token для доступа к API IGDB
import requests
import json
resp = requests.post('https://id.twitch.tv/oauth2/token', data={
    'client_id': 'po0syvw9g7dh0et8fdhse6y12l4gmu',
    'client_secret': '67676767766767767',
    'grant_type': 'client_credentials'
})

data = resp.json()
# data

In [205]:
# data

{'access_token': 'сиксевен676767',
 'expires_in': 5283769,
 'token_type': 'bearer'}

In [ ]:
# os.environ['access_token'] = '67'
# os.getenv('access_token')

В С Ë. Мы получили токен для работы с API. Дальше будем работать напрямую с докой и искать данные

Так как по условию нам нужно использовать не менее 5 эндпойнтов — было принято решение написать функцию


### Всю последующую работу я делаю опираясь на Seminar 8 - API

Далее сделаем request. В основном из методов в документации только post. Обращаемся к `https://api.igdb.com/v4`. 

Еще я добавил логгирование чтобы отслеживать важные моменты и отлавливать места с проблемаии

In [26]:
import logging 
logging.basicConfig(level=logging.INFO)

In [62]:
token = os.getenv('access_token')
# token
# cid


In [ ]:
# этот url нам понадобится для запросов к API IGDB
url = 'https://api.igdb.com/v4/games'

# в headers идентифицируем себя и указываем, что хотим получить json
headers = {
    'Client-ID': 'po0syvw9g7dh0et8fdhse6y12l4gmu',
    'Authorization': f"Bearer sixseven",
    'Accept': 'application/json'
}

# а теперь делаем запрос к API IGDB, чтобы получить 10 игр с их рейтингами
body = 'fields name, rating; limit 10;'

logging.info("Отправляем запрос к API IGDB...")
#добавил еще таймаут, чтобы не зависало на запросе
games10 = requests.post(url, headers=headers, data=body, timeout=10)
logging.info(f'Ответ от IGDB {games10.status_code}')

INFO:root:Отправляем запрос к API IGDB...
INFO:root:Ответ от IGDB 200


УРААААА!!! Ответ 200, у нас есть data. Давайте посмотрим как там внутри

In [67]:
games10.json()

[{'id': 350392, 'name': 'Rival Species'},
 {'id': 169876, 'name': 'Timothy and the Tower of Mu'},
 {'id': 207571, 'name': 'A Very Corporate Escape Room'},
 {'id': 339266, 'name': 'Power Guy World'},
 {'id': 63844, 'name': 'Ace wo Nerae!', 'rating': 52.90462943179914},
 {'id': 371149, 'name': 'The Deal'},
 {'id': 330684, 'name': 'Nightmare Kart: The Old Karts'},
 {'id': 129477, 'name': 'Chomp Chomp'},
 {'id': 179243,
  'name': 'Satella-Q: Mou Sugu Haru desu ne - Chotto Sate-Q Shimasen ka?'},
 {'id': 152887, 'name': 'BanG Dream! Girls Band Party! for Nintendo Switch'}]

Обратим внимание, что рейтинг есть не везде. Покопавшись чуть глубже в доке узнал, что поле с рейтингом является опциональном. Вообще, покопавшись в метриках можно заметить, что там целый набор метрик: rating, rating_count, aggregated_rating, aggregated_rating_count, total_rating и total_rating_count. 

В таком случае имеет место быть заселектить разные поля, а самое главное поставить фильтры... Я не до конца понимаю по какому принципу сортируется сейчас, поэтому лучше добавить фильтры и сортировку...

In [ ]:
# сделаем функцию чтобы постоянно не писать эти заголовки и url
def igdb_request(body, timeout=10, endpoint= 'https://api.igdb.com/v4/games'):
    # этот url нам понадобится для запросов к API IGDB
    url = endpoint

    # в headers идентифицируем себя и указываем, что хотим получить json
    headers = {
        'Client-ID': 'po0syvw9g7dh0et8fdhse6y12l4gmu',
        'Authorization': f"Bearer 67",
        'Accept': 'application/json'
    }

    logging.info(f"Отправляем запрос к API IGDB {endpoint} ...")
    data_req = requests.post(url, headers=headers, data=body, timeout=timeout)
    logging.info(f'Ответ от IGDB {endpoint}: {data_req.status_code}')

    return data_req.json()

In [244]:
igdb_request('fields *; limit 10;', endpoint='https://api.igdb.com/v4/game_time_to_beats')

INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/game_time_to_beats ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/game_time_to_beats: 200


[{'id': 3575,
  'game_id': 201711,
  'hastily': 14400,
  'normally': 28800,
  'completely': 32400,
  'count': 1,
  'created_at': 1731046407,
  'updated_at': 1731046407,
  'checksum': 'b7628541-1f60-8cfd-667f-7da7c20a9f6d'},
 {'id': 5262,
  'game_id': 293842,
  'count': 1,
  'created_at': 1751576095,
  'updated_at': 1751576095,
  'checksum': 'bf4dfe07-d5cb-c182-17bc-893b61cfb939'},
 {'id': 3609,
  'game_id': 121835,
  'completely': 3600,
  'count': 1,
  'created_at': 1731936621,
  'updated_at': 1731936621,
  'checksum': 'a642c58f-be03-a110-8936-332fff5cc3a7'},
 {'id': 1025,
  'game_id': 68142,
  'hastily': 3600,
  'normally': 3720,
  'completely': 7200,
  'count': 1,
  'created_at': 1728556561,
  'updated_at': 1728556561,
  'checksum': 'bb6928ff-5dae-c4fb-9d25-368303322ea7'},
 {'id': 3028,
  'game_id': 275404,
  'hastily': 3600,
  'normally': 7200,
  'completely': 5400,
  'count': 3,
  'created_at': 1728556620,
  'updated_at': 1772832147,
  'checksum': 'a5883ac7-4cf3-82c2-cf49-52365a310

Ну и дополним списочком всех эндпойнтов, которые у нас есть

In [358]:
endpoints = {
    'games': 'https://api.igdb.com/v4/games',
    # 'companies': 'https://api.igdb.com/v4/companies',
    # 'game_engines': 'https://api.igdb.com/v4/game_engines',
    # 'genres': 'https://api.igdb.com/v4/genres',
    'involved_companies': 'https://api.igdb.com/v4/involved_companies',
    # 'platforms': 'https://api.igdb.com/v4/platforms',
    'keywords': 'https://api.igdb.com/v4/keywords',
    # 'franchises': 'https://api.igdb.com/v4/franchises',
    'gttb': 'https://api.igdb.com/v4/game_time_to_beats',
    'alternative_names': 'https://api.igdb.com/v4/alternative_names'
}

In [76]:
endpoints['games']

'https://api.igdb.com/v4/games'

### СЕЛЕКТИМ ВСЕ
Селектим все данные, где есть рейтинг и количество оценок >20


In [97]:
!pip install tqdm

In [98]:
from tqdm import tqdm
import time 

#### Селект игр

In [246]:
body_games_sample='''
fields
    id,
    name,
    slug,
    first_release_date,
    platforms.name,
    genres.name,
    themes.name,
    game_modes.name,
    game_type,
    involved_companies.company.name,
    rating,
    rating_count,
    aggregated_rating,
    aggregated_rating_count,
    total_rating,
    total_rating_count,
    hypes,
    external_games.uid,
    external_games.external_game_source,
    franchise,
    parent_game,
    remakes,
    remasters,
    storyline,
    summary,
    keywords;
where rating!=null & rating_count>20;
sort rating desc;
limit 500;
'''
data_games_raw_sample = igdb_request(body_games_sample, endpoint=endpoints['games'])

INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/games ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/games: 200


In [88]:
#сэмпл

import pandas as pd 
samples = pd.DataFrame(data_games_raw)
samples

,id,external_games,first_release_date,game_modes,genres,involved_companies,keywords,name,platforms,rating,...,total_rating_count,game_type,storyline,aggregated_rating,aggregated_rating_count,hypes,parent_game,franchise,remakes,remasters
0,65755,"[{'id': 1896086, 'uid': '32308', 'external_gam...",1.311898e+09,"[{'id': 1, 'name': 'Single player'}]","[{'id': 2, 'name': 'Point-and-click'}, {'id': ...","[{'id': 55465, 'company': {'id': 13608, 'name'...","[72, 2050, 4158]",Trailer Park King,"[{'id': 12, 'name': 'Xbox 360'}]",100.000000,...,47,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,112659,"[{'id': 1929804, 'uid': 'B083T65GQX', 'externa...",1.543277e+09,"[{'id': 1, 'name': 'Single player'}]","[{'id': 25, 'name': 'Hack and slash/Beat 'em u...","[{'id': 107496, 'company': {'id': 14055, 'name...","[1868, 2777, 4142, 4310]",Batman: Arkham Collection,"[{'id': 48, 'name': 'PlayStation 4'}, {'id': 6...",99.566526,...,25,3,Batman: Arkham Asylum\nThe game begins with Ba...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20196,"[{'id': 1933251, 'uid': 'B00CE5K2M0', 'externa...",1.373328e+09,"[{'id': 1, 'name': 'Single player'}]","[{'id': 5, 'name': 'Shooter'}, {'id': 24, 'nam...","[{'id': 37443, 'company': {'id': 129, 'name': ...",[25891],Metal Gear Solid: The Legacy Collection,"[{'id': 9, 'name': 'PlayStation 3'}]",99.513151,...,45,3,NaN,87.000000,2.0,NaN,NaN,NaN,NaN,NaN
3,47304,"[{'id': 146226, 'uid': '11837', 'external_game...",1.103674e+09,NaN,"[{'id': 10, 'name': 'Racing'}]",NaN,"[4142, 4205]",American Chopper,"[{'id': 11, 'name': 'Xbox'}, {'id': 6, 'name':...",99.457627,...,49,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,45131,"[{'id': 1933044, 'uid': 'B00CY92XU0', 'externa...",1.379376e+09,"[{'id': 1, 'name': 'Single player'}]","[{'id': 5, 'name': 'Shooter'}, {'id': 10, 'nam...","[{'id': 104254, 'company': {'id': 139, 'name':...",NaN,Grand Theft Auto V: Special Edition,"[{'id': 9, 'name': 'PlayStation 3'}, {'id': 12...",99.310093,...,81,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,19565,"[{'id': 1935192, 'uid': 'B07CNDV7J4', 'externa...",1.536278e+09,"[{'id': 1, 'name': 'Single player'}]","[{'id': 25, 'name': 'Hack and slash/Beat 'em u...","[{'id': 36449, 'company': {'id': 10100, 'name'...","[1401, 2387, 3540, 3934, 4142, 12083, 12087, 1...",Marvel's Spider-Man,"[{'id': 48, 'name': 'PlayStation 4'}]",87.191812,...,2034,0,NaN,89.687500,16.0,68.0,NaN,NaN,NaN,[138949]
496,127816,"[{'id': 2069628, 'uid': '2125225264', 'externa...",1.599178e+09,"[{'id': 1, 'name': 'Single player'}]","[{'id': 9, 'name': 'Puzzle'}, {'id': 12, 'name...","[{'id': 106604, 'company': {'id': 18629, 'name...","[72, 225, 962, 1313, 1438, 1669, 3534, 4484]",Paradise Killer,"[{'id': 169, 'name': 'Xbox Series X|S'}, {'id'...",87.180497,...,56,0,NaN,79.111111,9.0,NaN,NaN,NaN,NaN,NaN
497,8195,"[{'id': 145626, 'uid': '12494', 'external_game...",7.693056e+08,"[{'id': 1, 'name': 'Single player'}]","[{'id': 5, 'name': 'Shooter'}, {'id': 8, 'name...","[{'id': 19645, 'company': {'id': 1238, 'name':...","[69, 103, 129, 575, 1075, 1097, 4142, 4310, 46...",RoboCop Versus the Terminator,"[{'id': 29, 'name': 'Sega Mega Drive/Genesis'}]",87.180496,...,22,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
498,328145,"[{'id': 2994911, 'uid': '1185686433', 'externa...",1.271290e+09,"[{'id': 2, 'name': 'Multiplayer'}]","[{'id': 7, 'name': 'Music'}, {'id': 32, 'name'...","[{'id': 303561, 'company': {'id': 20241, 'name...","[358, 1496]",Idate: Online Interactive Dating,"[{'id': 6, 'name': 'PC (Microsoft Windows)'}]",87.142857,...,21,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [247]:
body_games=[]

for offset in tqdm(range(0, 30000,500)):
    body=f'''
    fields
        id,
        name,
        slug,
        first_release_date,
        platforms.name,
        genres.name,
        themes.name,
        game_modes.name,
        game_type,
        involved_companies.company.name,
        rating,
        rating_count,
        aggregated_rating,
        aggregated_rating_count,
        total_rating,
        total_rating_count,
        hypes,
        external_games.uid,
        external_games.external_game_source,
        franchise,
        parent_game,
        remakes,
        remasters,
        keywords;
    where rating_count>0;
    sort rating_count desc;
    limit 500;
    offset {offset};
    '''
    
    batch=igdb_request(body, endpoint=endpoints['games'])
    
    if not batch:
        logging.info(f'на offset={offset} данные закончились')
        break
    
    body_games.extend(batch)
    time.sleep(0.33) # IGDB позволяет делать 4 запроса в секунду, так что добавляем паузу для успокоения

logging.info(f'Всего игр получено: {len(body_games)}')

  0%|          | 0/60 [00:00<?, ?it/s]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/games ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/games: 200
  2%|▏         | 1/60 [00:02<02:13,  2.26s/it]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/games ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/games: 200
  3%|▎         | 2/60 [00:04<01:58,  2.04s/it]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/games ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/games: 200
  5%|▌         | 3/60 [00:06<01:59,  2.09s/it]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/games ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/games: 200
  7%|▋         | 4/60 [00:08<02:08,  2.30s/it]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/games ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/games: 200
  8%|▊         | 5/60 [00:11<02:15,  2.46s/it]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/games ...

In [248]:
df_games = pd.DataFrame(body_games)
df_games

,id,aggregated_rating,aggregated_rating_count,external_games,first_release_date,game_modes,genres,involved_companies,keywords,name,...,slug,themes,total_rating,total_rating_count,game_type,hypes,remasters,remakes,franchise,parent_game
0,1020,88.137931,27.0,"[{'id': 1935351, 'uid': 'B00NOP3QF4', 'externa...",1.379376e+09,"[{'id': 1, 'name': 'Single player'}, {'id': 2,...","[{'id': 5, 'name': 'Shooter'}, {'id': 10, 'nam...","[{'id': 20018, 'company': {'id': 139, 'name': ...","[3, 21, 22, 25, 30, 57, 64, 72, 109, 129, 153,...",Grand Theft Auto V,...,grand-theft-auto-v,"[{'id': 1, 'name': 'Action'}, {'id': 27, 'name...",88.874560,5654,0,NaN,NaN,NaN,NaN,NaN
1,1942,91.730769,26.0,"[{'id': 1935171, 'uid': 'B00T3SPV36', 'externa...",1.431994e+09,"[{'id': 1, 'name': 'Single player'}]","[{'id': 12, 'name': 'Role-playing (RPG)'}, {'i...","[{'id': 17436, 'company': {'id': 50, 'name': '...","[98, 129, 138, 151, 226, 227, 482, 537, 592, 6...",The Witcher 3: Wild Hunt,...,the-witcher-3-wild-hunt,"[{'id': 1, 'name': 'Action'}, {'id': 17, 'name...",92.760666,5217,0,179.0,NaN,NaN,NaN,NaN
2,72,92.444444,9.0,"[{'id': 189642, 'uid': 'UCnj3zKNoYGZLWzFAo-bB3...",1.303085e+09,"[{'id': 1, 'name': 'Single player'}, {'id': 2,...","[{'id': 8, 'name': 'Platform'}, {'id': 9, 'nam...","[{'id': 265481, 'company': {'id': 56, 'name': ...","[575, 592, 962, 1158, 1181, 1293, 1440, 2071, ...",Portal 2,...,portal-2,"[{'id': 1, 'name': 'Action'}, {'id': 18, 'name...",91.889774,4280,0,NaN,NaN,NaN,NaN,NaN
3,472,79.916667,10.0,"[{'id': 1933029, 'uid': 'B0050SYNYQ', 'externa...",1.320883e+09,"[{'id': 1, 'name': 'Single player'}]","[{'id': 12, 'name': 'Role-playing (RPG)'}, {'i...","[{'id': 3719, 'company': {'id': 126, 'name': '...","[22, 96, 129, 151, 159, 211, 221, 226, 236, 37...",The Elder Scrolls V: Skyrim,...,the-elder-scrolls-v-skyrim,"[{'id': 1, 'name': 'Action'}, {'id': 17, 'name...",83.804026,4171,0,NaN,[19457],NaN,NaN,NaN
4,732,93.142857,6.0,"[{'id': 1933194, 'uid': 'B0183O87HM', 'externa...",1.098749e+09,"[{'id': 1, 'name': 'Single player'}, {'id': 3,...","[{'id': 5, 'name': 'Shooter'}, {'id': 10, 'nam...","[{'id': 55979, 'company': {'id': 365, 'name': ...","[21, 57, 58, 72, 109, 129, 155, 221, 222, 284,...",Grand Theft Auto: San Andreas,...,grand-theft-auto-san-andreas,"[{'id': 1, 'name': 'Action'}, {'id': 23, 'name...",91.585342,3866,0,2.0,[178126],NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24002,2806,NaN,NaN,"[{'id': 213231, 'uid': 'xbox36066acd000-77fe-1...",1.318896e+09,"[{'id': 1, 'name': 'Single player'}]","[{'id': 10, 'name': 'Racing'}]","[{'id': 179773, 'company': {'id': 1315, 'name'...","[997, 1293, 4134, 4142, 4245, 4523, 5328, 5783]",Ben 10: Galactic Racing,...,ben-10-galactic-racing,"[{'id': 1, 'name': 'Action'}, {'id': 18, 'name...",70.000000,1,0,NaN,NaN,NaN,101.0,NaN
24003,2804,NaN,NaN,"[{'id': 1923350, 'uid': '313251', 'external_ga...",1.374710e+09,"[{'id': 1, 'name': 'Single player'}]","[{'id': 15, 'name': 'Strategy'}]","[{'id': 47829, 'company': {'id': 10075, 'name'...","[151, 1158, 1494, 4004, 4134, 4544, 5212]",Citadels,...,citadels,"[{'id': 1, 'name': 'Action'}, {'id': 22, 'name...",20.000000,1,0,NaN,NaN,NaN,NaN,NaN
24004,2764,NaN,NaN,"[{'id': 132371, 'uid': '27251', 'external_game...",1.035936e+09,"[{'id': 1, 'name': 'Single player'}, {'id': 4,...",NaN,"[{'id': 7366, 'company': {'id': 1014, 'name': ...","[1075, 4142, 9579]",Nickelodeon Party Blast,...,nickelodeon-party-blast,"[{'id': 40, 'name': 'Party'}]",38.000000,1,0,NaN,NaN,NaN,NaN,NaN
24005,1554,NaN,NaN,"[{'id': 137368, 'uid': '21684', 'external_game...",1.093478e+09,"[{'id': 1, 'name': 'Single player'}]","[{'id': 4, 'name': 'Fighting'}, {'id': 12, 'na...","[{'id': 10959, 'company': {'id': 321, 'name': ...","[1423, 2152, 6094, 7152, 9777]",Virtua Quest,...,virtua-quest,"[{'id': 1, 'name': 'Action'}, {'id': 18, 'name...",34.000000,1,0,NaN,NaN,NaN,NaN,NaN


Итак... Мало того, что многие ячейки остались внутри json-ами(это поправимо на постобработке eda), так мы еще и увидели, что заселектилось всего 10 ячеек. Опытным путем и игрой с лимитами, выяснялось — что нормально селектятся 500- строк...

Изучая проблему, я осознал, что мы не можем полностью заселектить все. Но можем селектить БАТЧАМИ... О том как это делается я прочитал в [статье на proglib](https://proglib.io/p/paketnyy-api-obedinenie-zaprosov-s-pomoshchyu-asyncio-i-batch-api-2023-03-23).

Еще я добавил TQDM, чтобы мы могли отслеживать сколько времени осталось и на каком мы этапе...
Берем данные от самых больших по количеству отзывов, до самых маленьких



#### Селект вовлеченных компаний 


Для нас важно заселектить только те компании, которые уже есть в выборке чтобы потом просто, возможно, сделать left join

делаем ПО-УМНОМУ

In [302]:
game_ids = df_games['id'].tolist()
chunk_size = 400
involved_companies = []

In [303]:
len(game_ids)

24007

In [304]:
for i in tqdm(range(0, len(game_ids), chunk_size)):
    chunk = game_ids[i:i + chunk_size]
    ids_str = ','.join(map(str, chunk))
    logging.debug(f'взяты ids_str: {ids_str}')
    body = f'''
    fields
        company.name,
        company.id,
        game.name,
        game.id;
    where game=({ids_str}) & developer=true;
    limit 500;
    '''
    
    batch = igdb_request(body, endpoint=endpoints['involved_companies'])
    
    if not batch:
        logging.info(f'на чанке {i} данные пустые')
        continue
        
    involved_companies.extend(batch)
    time.sleep(0.33)

logging.info(f'Всего involved-компаний получено: {len(involved_companies)}')

  0%|          | 0/61 [00:00<?, ?it/s]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/involved_companies ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/involved_companies: 200
  2%|▏         | 1/61 [00:01<01:09,  1.16s/it]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/involved_companies ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/involved_companies: 200
  3%|▎         | 2/61 [00:01<00:56,  1.05it/s]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/involved_companies ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/involved_companies: 200
  5%|▍         | 3/61 [00:02<00:56,  1.03it/s]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/involved_companies ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/involved_companies: 200
  7%|▋         | 4/61 [00:04<01:00,  1.07s/it]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/involved_companies ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/involved_com

In [306]:
df_involved_companies = pd.DataFrame(involved_companies)
df_involved_companies

,id,company,game
0,8,"{'id': 7, 'name': 'Visceral Games'}","{'id': 38, 'name': 'Dead Space 2'}"
1,747,"{'id': 203, 'name': 'Illusion Softworks'}","{'id': 39, 'name': 'Mafia'}"
2,108197,"{'id': 552, 'name': 'Crystal Dynamics'}","{'id': 37777, 'name': 'Shadow of the Tomb Raid..."
3,4483,"{'id': 30, 'name': 'Team Bondi'}","{'id': 109, 'name': 'L.A. Noire'}"
4,9853,"{'id': 38, 'name': 'Ubisoft Montreal'}","{'id': 1266, 'name': 'Assassin's Creed III'}"
...,...,...,...
25034,7360,"{'id': 1239, 'name': 'Gorilla Systems Corporat...","{'id': 2849, 'name': 'All Star Cheer Squad'}"
25035,7366,"{'id': 1014, 'name': 'Data Design Interactive'}","{'id': 2764, 'name': 'Nickelodeon Party Blast'}"
25036,10959,"{'id': 321, 'name': 'Tose'}","{'id': 1554, 'name': 'Virtua Quest'}"
25037,2513,"{'id': 745, 'name': 'Futuremark Games Studio'}","{'id': 1057, 'name': 'Unstoppable Gorg'}"


делегировано глебу для EDA

#### Селект ключевых слов по играм

In [324]:
keywords_per_every_game = df_games[['id', 'keywords']].explode('keywords')
keywords_per_every_game

,id,keywords
0,1020,3
0,1020,21
0,1020,22
0,1020,25
0,1020,30
...,...,...
24006,1057,5814
24006,1057,6539
24006,1057,7417
24006,1057,11533


In [325]:
keyword_counts = keywords_per_every_game.groupby('keywords').count().sort_values('id', ascending=False).reset_index().rename(columns={'id': 'game_count'})
keyword_counts

,keywords,game_count
0,4134,3861
1,1158,2705
2,4359,1843
3,4004,1672
4,1293,1633
...,...,...
4401,42056,1
4402,42075,1
4403,42082,1
4404,42100,1


In [337]:
keywords_ids = keyword_counts['keywords'].tolist()
chunk_size = 400
keywords_per_every_game_api = []

In [338]:
len(keywords_ids)

4406

In [339]:
for i in tqdm(range(0, len(keywords_ids), chunk_size)):
    chunk = keywords_ids[i:i + chunk_size]
    ids_str = ','.join(map(str, chunk))
    logging.debug(f'взяты ids_str: {ids_str}')
    body = f'''
    fields
        id,
        name,
        slug;
    where id=({ids_str});
    limit 500;
    '''
    
    batch = igdb_request(body, endpoint=endpoints['keywords'])
    
    if not batch:
        logging.info(f'на чанке {i} данные пустые')
        continue
        
    keywords_per_every_game_api.extend(batch)
    time.sleep(0.33)

logging.info(f'Всего keywords получено: {len(keywords_per_every_game_api)}')

  0%|          | 0/12 [00:00<?, ?it/s]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/keywords ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/keywords: 200
  8%|▊         | 1/12 [00:01<00:11,  1.04s/it]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/keywords ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/keywords: 200
 17%|█▋        | 2/12 [00:02<00:10,  1.05s/it]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/keywords ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/keywords: 200
 25%|██▌       | 3/12 [00:02<00:08,  1.10it/s]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/keywords ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/keywords: 200
 33%|███▎      | 4/12 [00:03<00:07,  1.05it/s]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/keywords ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/keywords: 200
 42%|████▏     | 5/12 [00:04<00:06,  1.13it/s]INFO:root:Отправляем запрос к API IGDB htt

In [340]:
df_keywords = pd.DataFrame(keywords_per_every_game_api)
df_keywords

,id,name,slug
0,4613,non-player character,non-player-character
1,1971,over the shoulder,over-the-shoulder
2,4410,game reference,game-reference
3,497,summoning support,summoning-support
4,669,frog,frog
...,...,...,...
4401,42056,scanner,scanner
4402,42075,giant,giant
4403,42082,paper plane,paper-plane
4404,42100,drill,drill


#### Селект ключевых слов по time to beat

In [356]:
game_ids
chunk_size = 400
times_to_beat = []

In [357]:
for i in tqdm(range(0, len(game_ids), chunk_size)):
    chunk = game_ids[i:i + chunk_size]
    ids_str = ','.join(map(str, chunk))
    logging.debug(f'взяты ids_str: {ids_str}')
    body = f'''
    fields
        game_id,
        hastily,
        normally,
        completely,
        count;
    where game_id=({ids_str});
    limit 500;
    '''
    
    batch = igdb_request(body, endpoint=endpoints['gttb'])
    
    if not batch:
        logging.info(f'на чанке {i} данные пустые')
        continue
        
    times_to_beat.extend(batch)
    time.sleep(0.33)

logging.info(f'Всего времен прохождения по играм получено: {len(times_to_beat)}')

  0%|          | 0/61 [00:00<?, ?it/s]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/game_time_to_beats ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/game_time_to_beats: 200
  2%|▏         | 1/61 [00:01<01:00,  1.01s/it]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/game_time_to_beats ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/game_time_to_beats: 200
  3%|▎         | 2/61 [00:02<00:59,  1.01s/it]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/game_time_to_beats ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/game_time_to_beats: 200
  5%|▍         | 3/61 [00:03<00:59,  1.02s/it]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/game_time_to_beats ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/game_time_to_beats: 200
  7%|▋         | 4/61 [00:03<00:52,  1.08it/s]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/game_time_to_beats ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/game_time_to

In [355]:
df_gttb = pd.DataFrame(times_to_beat)
df_gttb

,id,game_id,hastily,normally,completely,count
0,2033,55163,35280.0,45086.0,55800.0,10
1,1001,26758,21600.0,64260.0,246700.0,8
2,29,7444,9600.0,27200.0,36000.0,7
3,18,5025,NaN,13500.0,19800.0,11
4,1038,29004,21600.0,41670.0,43104.0,11
...,...,...,...,...,...,...
717,8201,2178,7200.0,NaN,NaN,1
718,8315,2687,NaN,NaN,1260000.0,1
719,8354,981,39600.0,50400.0,79200.0,1
720,8371,1557,NaN,64800.0,NaN,1


#### Селект альтернативных названий

In [367]:
game_ids
chunk_size = 400
alt_names = []

In [368]:
for i in tqdm(range(0, len(game_ids), chunk_size)):
    chunk = game_ids[i:i + chunk_size]
    ids_str = ','.join(map(str, chunk))
    logging.debug(f'взяты ids_str: {ids_str}')
    body = f'''
    fields
        comment,
        game,
        name;
    where game=({ids_str});
    limit 500;
    '''
    
    batch = igdb_request(body, endpoint=endpoints['alternative_names'])
    
    if not batch:
        logging.info(f'на чанке {i} данные пустые')
        continue
        
    alt_names.extend(batch)
    time.sleep(0.33)

logging.info(f'Всего альтернативных названий получено: {len(alt_names)}')

  0%|          | 0/61 [00:00<?, ?it/s]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/alternative_names ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/alternative_names: 200
  2%|▏         | 1/61 [00:01<01:07,  1.12s/it]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/alternative_names ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/alternative_names: 200
  3%|▎         | 2/61 [00:02<01:22,  1.39s/it]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/alternative_names ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/alternative_names: 200
  5%|▍         | 3/61 [00:03<01:13,  1.28s/it]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/alternative_names ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/alternative_names: 200
  7%|▋         | 4/61 [00:04<01:08,  1.19s/it]INFO:root:Отправляем запрос к API IGDB https://api.igdb.com/v4/alternative_names ...
INFO:root:Ответ от IGDB https://api.igdb.com/v4/alternative_names: 20

In [369]:
df_alt_names = pd.DataFrame(alt_names)
df_alt_names

,id,comment,game,name
0,32932,Japanese title - romanization,1030,Zelda no Densetsu: Majora no Kamen
1,18198,Other,342,Duke3D
2,2974,Working title,673,Doom: Evil Unleashed
3,2997,Taiwanese title,281,重返德軍總部
4,22244,Other,133,War3:TFT
...,...,...,...,...
28233,52848,European title,2849,All Star Cheerleader
28234,62332,Japanese title - romanization,1554,Virtua Fighter: Cyber Generation ~Judgement Si...
28235,217324,Windows Executable,2938,UDK.exe
28236,249531,Windows Executable,1057,unstoppable_gorg.exe


---
### добавим все это в csv формат чтобы глебу не пришлось перепарсивать


In [370]:
df_games.to_pickle('df_games.pkl')
df_involved_companies.to_pickle('df_involved_companies.pkl')
df_keywords.to_pickle('df_keywords.pkl')
df_gttb.to_pickle('df_gttb.pkl')
df_alt_names.to_pickle('df_alt_names.pkl')
